# **1. Perkenalan Dataset**

## Wine Quality Dataset (Red Wine)

Dataset ini berasal dari **UCI Machine Learning Repository** dan berisi informasi tentang varian merah dari Portuguese "Vinho Verde" wine.

### Informasi Dataset:
- **Sumber**: [UCI ML Repository - Wine Quality](https://archive.ics.uci.edu/ml/datasets/wine+quality)
- **Jumlah Sampel**: 1,599
- **Jumlah Fitur**: 11 fitur numerik + 1 target
- **Target**: `quality` (skor 0-10, pada dataset ini berkisar 3-8)

### Fitur-fitur:
| No | Fitur | Deskripsi |
|---|---|---|
| 1 | fixed acidity | Kadar asam tetap |
| 2 | volatile acidity | Kadar asam mudah menguap |
| 3 | citric acid | Kadar asam sitrat |
| 4 | residual sugar | Gula sisa setelah fermentasi |
| 5 | chlorides | Kadar klorida |
| 6 | free sulfur dioxide | SO2 bebas |
| 7 | total sulfur dioxide | Total SO2 |
| 8 | density | Kepadatan |
| 9 | pH | Tingkat keasaman |
| 10 | sulphates | Kadar sulfat |
| 11 | alcohol | Kadar alkohol |

### Tujuan:
Membangun model klasifikasi untuk memprediksi kualitas wine berdasarkan sifat fisikokimia (quality dikategorikan menjadi: **low**, **medium**, **high**).

# **2. Import Library**

Mengimpor pustaka yang dibutuhkan untuk analisis data, visualisasi, dan preprocessing.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

print('Libraries imported successfully!')

# **3. Memuat Dataset**

Memuat dataset Wine Quality (Red Wine) dari folder `winequality_raw`. Dataset menggunakan separator semicolon (`;`).

In [ ]:
# Load dataset
df = pd.read_csv('../winequality_raw/winequality-red.csv', sep=';')

# Tampilkan info dasar
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print()
df.head(10)

In [ ]:
# Info dataset
df.info()

In [ ]:
# Statistik deskriptif
df.describe()

# **4. Exploratory Data Analysis (EDA)**

Pada tahap ini, kita akan melakukan analisis eksplorasi data untuk memahami karakteristik dataset secara mendalam.

## 4.1 Cek Missing Values & Duplicates

In [ ]:
# Cek missing values
print('=== Missing Values ===')
print(df.isnull().sum())
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

print('\n=== Duplicate Rows ===')
print(f'Total duplicate rows: {df.duplicated().sum()}')

## 4.2 Distribusi Target (Quality)

In [ ]:
# Distribusi kelas target
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
quality_counts = df['quality'].value_counts().sort_index()
colors = sns.color_palette('viridis', len(quality_counts))
axes[0].bar(quality_counts.index, quality_counts.values, color=colors)
axes[0].set_xlabel('Quality Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Wine Quality Scores')

# Percentage
for i, (idx, val) in enumerate(zip(quality_counts.index, quality_counts.values)):
    axes[0].text(idx, val + 5, f'{val}\n({val/len(df)*100:.1f}%)', ha='center', fontsize=9)

# Pie chart
axes[1].pie(quality_counts.values, labels=[f'Quality {q}' for q in quality_counts.index],
            autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Wine Quality Distribution')

plt.tight_layout()
plt.show()

print(f'Quality range: {df["quality"].min()} - {df["quality"].max()}')
print(f'Most common quality: {df["quality"].mode()[0]}')

## 4.3 Distribusi Fitur

In [ ]:
# Distribusi setiap fitur (histogram)
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
axes = axes.flatten()

for i, col in enumerate(df.columns):
    axes[i].hist(df[col], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(col, fontweight='bold')
    axes[i].axvline(df[col].mean(), color='red', linestyle='--', label=f'Mean: {df[col].mean():.2f}')
    axes[i].axvline(df[col].median(), color='green', linestyle='--', label=f'Median: {df[col].median():.2f}')
    axes[i].legend(fontsize=8)

# Hide the last empty subplot
axes[-1].set_visible(False)

plt.suptitle('Feature Distributions', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 4.4 Korelasi Antar Fitur

In [ ]:
# Correlation heatmap
plt.figure(figsize=(14, 10))

corr_matrix = df.corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5,
            cbar_kws={'label': 'Correlation Coefficient'})

plt.title('Correlation Heatmap - Wine Quality Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Korelasi dengan target
print('\n=== Correlation with Quality ===')
corr_with_target = corr_matrix['quality'].drop('quality').sort_values(ascending=False)
print(corr_with_target)

## 4.5 Boxplot untuk Deteksi Outlier

In [ ]:
# Boxplot untuk setiap fitur
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
axes = axes.flatten()

feature_cols = [col for col in df.columns if col != 'quality']

for i, col in enumerate(feature_cols):
    sns.boxplot(data=df, y=col, ax=axes[i], color='lightcoral')
    axes[i].set_title(col, fontweight='bold')
    
    # Count outliers using IQR
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum()
    axes[i].set_xlabel(f'Outliers: {outliers}', fontsize=10)

axes[-1].set_visible(False)

plt.suptitle('Boxplots - Outlier Detection', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 4.6 Fitur vs Quality

In [ ]:
# Boxplot setiap fitur berdasarkan quality
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
axes = axes.flatten()

for i, col in enumerate(feature_cols):
    sns.boxplot(data=df, x='quality', y=col, ax=axes[i], palette='viridis')
    axes[i].set_title(f'{col} vs Quality', fontweight='bold')

axes[-1].set_visible(False)

plt.suptitle('Features vs Wine Quality', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# **5. Data Preprocessing**

Tahapan preprocessing untuk mempersiapkan data agar siap digunakan untuk pelatihan model machine learning.

## 5.1 Handling Missing Values

In [ ]:
# Cek missing values (sudah dilakukan di EDA, konfirmasi)
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nTotal missing: {df.isnull().sum().sum()}')

# Jika ada missing values, handle
if df.isnull().sum().sum() > 0:
    # Fill numeric columns with median
    for col in df.select_dtypes(include=[np.number]).columns:
        if df[col].isnull().sum() > 0:
            df[col].fillna(df[col].median(), inplace=True)
            print(f'  Filled {col} with median')
else:
    print('\n✅ No missing values - no action needed!')

## 5.2 Removing Duplicates

In [ ]:
# Hapus duplikat
print(f'Shape before: {df.shape}')
duplicates = df.duplicated().sum()
print(f'Duplicate rows: {duplicates}')

if duplicates > 0:
    df = df.drop_duplicates()
    print(f'Shape after removing duplicates: {df.shape}')
    print(f'✅ Removed {duplicates} duplicate rows')
else:
    print('✅ No duplicates found')

## 5.3 Outlier Handling (IQR Method)

In [ ]:
# Handle outliers menggunakan IQR method (capping/clipping)
print('=== Outlier Handling (IQR Capping) ===')

feature_cols = [col for col in df.columns if col != 'quality']
outlier_summary = []

for col in feature_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    outliers_before = ((df[col] < lower) | (df[col] > upper)).sum()
    
    if outliers_before > 0:
        df[col] = df[col].clip(lower=lower, upper=upper)
        outlier_summary.append({'Feature': col, 'Outliers': outliers_before, 
                                'Lower': f'{lower:.4f}', 'Upper': f'{upper:.4f}'})

outlier_df = pd.DataFrame(outlier_summary)
if len(outlier_df) > 0:
    print(outlier_df.to_string(index=False))
    print(f'\n✅ Handled outliers in {len(outlier_df)} columns')
else:
    print('✅ No outliers detected')

## 5.4 Encoding Target Variable

In [ ]:
# Encode quality menjadi kategori
def quality_category(q):
    if q <= 4:
        return 'low'
    elif q <= 6:
        return 'medium'
    else:
        return 'high'

df['quality_label'] = df['quality'].apply(quality_category)

# Visualisasi distribusi setelah encoding
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Before encoding
df['quality'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Before Encoding (Original Quality)', fontweight='bold')
axes[0].set_xlabel('Quality Score')
axes[0].set_ylabel('Count')

# After encoding
order = ['low', 'medium', 'high']
df['quality_label'].value_counts().reindex(order).plot(kind='bar', ax=axes[1], 
    color=['#e74c3c', '#f39c12', '#2ecc71'])
axes[1].set_title('After Encoding (Categories)', fontweight='bold')
axes[1].set_xlabel('Quality Category')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print('\nDistribusi kategori:')
print(df['quality_label'].value_counts())

In [ ]:
# Label Encode
le = LabelEncoder()
df['quality_encoded'] = le.fit_transform(df['quality_label'])

print('Label Encoding Mapping:')
for cls, encoded in zip(le.classes_, le.transform(le.classes_)):
    print(f'  {cls} → {encoded}')

# Drop kolom quality asli dan quality_label
df = df.drop(columns=['quality', 'quality_label'])

print(f'\nDataset shape setelah encoding: {df.shape}')
df.head()

## 5.5 Feature Scaling (StandardScaler)

In [ ]:
# Scaling fitur menggunakan StandardScaler
feature_cols = [col for col in df.columns if col != 'quality_encoded']

scaler = StandardScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

print('✅ Features scaled using StandardScaler')
print(f'\nScaled features statistics:')
print(df[feature_cols].describe().round(3))

## 5.6 Train-Test Split

In [ ]:
# Split data
X = df.drop(columns=['quality_encoded'])
y = df['quality_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_test shape: {y_test.shape}')

print(f'\n=== Train Distribution ===')
print(y_train.value_counts().sort_index())
print(f'\n=== Test Distribution ===')
print(y_test.value_counts().sort_index())

## 5.7 Simpan Hasil Preprocessing

In [ ]:
import os

output_dir = 'winequality_preprocessing'
os.makedirs(output_dir, exist_ok=True)

# Gabungkan X dan y untuk disimpan
train_df = pd.concat([X_train.reset_index(drop=True), y_train.reset_index(drop=True)], axis=1)
test_df = pd.concat([X_test.reset_index(drop=True), y_test.reset_index(drop=True)], axis=1)
full_df = pd.concat([train_df, test_df], axis=0).reset_index(drop=True)

# Simpan
train_df.to_csv(os.path.join(output_dir, 'train.csv'), index=False)
test_df.to_csv(os.path.join(output_dir, 'test.csv'), index=False)
full_df.to_csv(os.path.join(output_dir, 'winequality_preprocessed.csv'), index=False)

print(f'✅ Data tersimpan di folder: {output_dir}/')
print(f'   - train.csv: {train_df.shape}')
print(f'   - test.csv: {test_df.shape}')
print(f'   - winequality_preprocessed.csv: {full_df.shape}')

# **Kesimpulan Eksperimen**

## Ringkasan:
1. **Dataset**: Wine Quality (Red Wine) — 1,599 sampel, 11 fitur
2. **Missing Values**: Tidak ada missing values
3. **Duplicates**: Ditemukan dan dihapus
4. **Outliers**: Ditangani menggunakan metode IQR (capping)
5. **Target Encoding**: Quality (3-8) → low/medium/high → label encoded
6. **Scaling**: StandardScaler diterapkan pada semua fitur
7. **Split**: 80% train, 20% test (stratified)

## Insight dari EDA:
- **Alcohol** memiliki korelasi positif tertinggi dengan quality
- **Volatile acidity** memiliki korelasi negatif tertinggi dengan quality
- Dataset memiliki **class imbalance** — mayoritas medium quality
- Beberapa fitur memiliki distribusi skewed (citric acid, free sulfur dioxide)

## Output:
- Data siap digunakan untuk training model di tahap selanjutnya
- File tersimpan di `winequality_preprocessing/`